<a href="https://colab.research.google.com/github/prathameshmowade/Patern-Recognition-/blob/main/PR_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PRACTICAL NO. 09 - FACE MASK DETECTION USING CNN
# Dataset:
# https://www.kaggle.com/datasets/omkargurav/face-mask-dataset
# ============================================================

!pip install -q kagglehub

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

# ============================================================
# 1. DOWNLOAD DATASET DIRECTLY FROM KAGGLE LINK
# ============================================================

path = kagglehub.dataset_download(
    "omkargurav/face-mask-dataset"
)

print("Dataset downloaded to:")
print(path)

# ============================================================
# 2. FIND TRAINING FOLDER AUTOMATICALLY
# ============================================================

train_path = None

# First, try the common 'data' subdirectory structure for Kaggle datasets
expected_data_path = os.path.join(path, 'data')
# Corrected to look for lowercase folder names: 'with_mask' and 'without_mask'
if os.path.exists(expected_data_path) and \
   os.path.exists(os.path.join(expected_data_path, "with_mask")) and \
   os.path.exists(os.path.join(expected_data_path, "without_mask")):
    train_path = expected_data_path
else:
    # Fallback to os.walk if 'data' subdirectory not found or incomplete
    print(f"Warning: 'with_mask' or 'without_mask' directories not found directly in {expected_data_path}. Attempting os.walk...")
    for root, dirs, files in os.walk(path):
        # Corrected to look for lowercase folder names: 'with_mask' and 'without_mask'
        if "with_mask" in dirs and "without_mask" in dirs:
            train_path = root
            break

if train_path:
    print("\nDataset folder:")
    print(train_path)
else:
    print("Error: Could not find the training data directory containing 'with_mask' and 'without_mask' folders.")

# ============================================================
# 3. IMAGE PREPROCESSING
# ============================================================

IMG_SIZE = 128
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20,
    rotation_range=20,
    zoom_range=0.20,
    shear_range=0.20,
    horizontal_flip=True
)

# Only proceed with flow_from_directory if train_path is valid
if train_path:
    train_data = datagen.flow_from_directory(
        train_path,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="binary",
        subset="training"
    )

    validation_data = datagen.flow_from_directory(
        train_path,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="binary",
        subset="validation",
        shuffle=False
    )

    print("\nClass Names:")
    print(train_data.class_indices)
else:
    print("Image preprocessing skipped: train_path not found.")
    # Initialize train_data and validation_data to None or empty objects to prevent further errors
    train_data = None
    validation_data = None


# ============================================================
# 4. SHOW SAMPLE IMAGES
# ============================================================

# Only proceed if train_data is available
if train_data:
    images, labels = next(train_data)

    plt.figure(figsize=(10, 8))

    for i in range(9):

        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i])

        if labels[i] == 0:
            plt.title("With Mask 😷")
        else:
            plt.title("Without Mask 🚫")

        plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Sample images display skipped: train_data not available.")


# ============================================================
# 5. CNN MODEL
# ============================================================

model = models.Sequential([

    layers.Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    ),

    layers.MaxPooling2D(2,2),

    layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D(2,2),

    layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        1,
        activation="sigmoid"
    )
])

model.summary()

# ============================================================
# 6. COMPILE MODEL
# ============================================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 7. TRAIN MODEL
# ============================================================

# Only train model if train_data and validation_data are available
if train_data and validation_data:
    history = model.fit(
        train_data,
        validation_data=validation_data,
        epochs=10
    )
else:
    print("Model training skipped: training or validation data not available.")
    history = None # Set history to None if training is skipped


# ============================================================
# 8. MODEL EVALUATION
# ============================================================

# Only evaluate model if validation_data is available
if validation_data:
    loss, accuracy = model.evaluate(validation_data)

    print("\n================================")
    print("MODEL PERFORMANCE")
    print("================================")

    print("Validation Loss     :", loss)
    print("Validation Accuracy :", accuracy * 100, "%")
else:
    print("Model evaluation skipped: validation data not available.")

# ============================================================
# 9. ACCURACY GRAPH
# ============================================================

# Only plot graphs if history is available
if history:
    plt.figure(figsize=(8,5))

    plt.plot(
        history.history["accuracy"],
        label="Training Accuracy"
    )

    plt.plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("CNN Training and Validation Accuracy")
    plt.legend()

    plt.show()
else:
    print("Accuracy graph skipped: model history not available.")


# ============================================================
# 10. LOSS GRAPH
# ============================================================

# Only plot graphs if history is available
if history:
    plt.figure(figsize=(8,5))

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("CNN Training and Validation Loss")
    plt.legend()

    plt.show()
else:
    print("Loss graph skipped: model history not available.")


# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

# Only generate report if validation_data is available
if validation_data:
    validation_data.reset()

    prediction = model.predict(validation_data)

    predicted_classes = (
        prediction > 0.5
    ).astype(int).flatten()

    actual_classes = validation_data.classes

    print("\n================================")
    print("CLASSIFICATION REPORT")
    print("================================")

    print(
        classification_report(
            actual_classes,
            predicted_classes,
            target_names=list(train_data.class_indices.keys())
        )
    )
else:
    print("Classification report skipped: validation data not available.")


# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

# Only generate confusion matrix if validation_data is available
if validation_data:
    cm = confusion_matrix(
        actual_classes,
        predicted_classes
    )

    print("\nConfusion Matrix:")
    print(cm)

    plt.figure(figsize=(6,5))

    plt.imshow(cm)

    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    class_names = list(train_data.class_indices.keys())

    plt.xticks([0,1], class_names)
    plt.yticks([0,1], class_names)

    for i in range(2):
        for j in range(2):

            plt.text(
                j,
                i,
                cm[i,j],
                ha="center",
                va="center"
            )

    plt.show()
else:
    print("Confusion matrix skipped: validation data not available.")


# ============================================================
# 13. SAVE MODEL
# ============================================================

# Only save model if training was performed (history is not None)
if history:
    model.save("Face_Mask_CNN.h5")
    print("\nModel saved as Face_Mask_CNN.h5")
else:
    print("Model save skipped: model was not trained.")

In [ ]:
import os

# Correcting the train_path based on common Kaggle dataset structure.
# The 'path' variable should be available from the previous cell.
if 'path' in locals():
    train_path = os.path.join(path, 'data')

    # Verify if the expected mask folders exist in this corrected path
    if not (os.path.exists(os.path.join(train_path, "WithMask")) and \
            os.path.exists(os.path.join(train_path, "WithoutMask"))):
        print(f"Warning: 'WithMask' or 'WithoutMask' directories not found in {train_path}")
        print("Attempting to search using os.walk as a fallback...")
        found_train_path_via_walk = None
        for root_walk, dirs_walk, files_walk in os.walk(path):
            if "WithMask" in dirs_walk and "WithoutMask" in dirs_walk:
                found_train_path_via_walk = root_walk
                break
        if found_train_path_via_walk:
            train_path = found_train_path_via_walk
        else:
            print("Fallback os.walk also failed to find the training path.")

    print("\nCorrected Dataset folder:")
    print(train_path)
else:
    print("Error: 'path' variable not found. Please ensure the previous cell runs successfully.")
